# 第十二章：共表达模块与基因集评分

**scRNA-seq 实践教程系列** | 基于 GSE136103 肝硬化数据集

---

## 写在前面

前面 11 篇我们一直在"单基因"层面分析数据：差异表达看的是每个基因、TF 分析看的是每个转录因子、轨迹分析看的是沿拟时序每个基因的变化趋势。

但生物学功能很少由单个基因决定。纤维化不是 COL1A1 一个基因的事，而是一整套胶原、基质重塑酶、TGF-β 信号通路成员协同工作的结果。 免疫细胞的杀伤功能也不是靠 GZMB 一个分子，而是 Perforin-Granzyme 通路的整体协调。

这一篇介绍两种"基因集"层面的分析思路：

共表达模块发现：数据驱动，让算法告诉你哪些基因总是一起表达
预定义基因集评分：假设驱动，用已知的功能通路基因集给每个细胞"打分"

一种是"盲挖"，一种是"有的放矢"——结合使用，才能既发现新模块，又验证已知通路。

## 为什么需要基因集评分

### 单基因分析的局限

假设你想知道哪些细胞在积极"促纤维化"。看 COL1A1 的表达？但 COL1A1 在肝细胞中也有基础表达。看 ACTA2？但 ACTA2 在平滑肌相关细胞中也高。单基因是"一叶障目"，基因集是"全貌"。

当我们把 COL1A1、COL1A2、COL3A1、ACTA2、TGFB1、CTGF 这一整套纤维化标志物打包成一个"评分"，就能更稳健地识别出真正在执行纤维化程序的细胞。

### 两种思路的对比

特性
共表达模块发现
预定义基因集评分

驱动方式
数据驱动（无监督）
假设驱动（先验知识）

需要基因列表？
❌ 自动发现
✅ 需要预先定义

优势
可发现未知共调控模块
可直接验证已知通路

典型工具
WGCNA, hdWGCNA
sc.tl.score_genes, AUCell

结果解读
需要功能注释
直接对应生物学功能

## Part A：共表达模块发现

### WGCNA 原理速览

WGCNA（Weighted Gene Co-expression Network Analysis）是 bulk RNA-seq 时代的经典方法，核心思想很简单：

计算所有基因对之间的相关系数
用"软阈值"（soft-thresholding）把相关系数转换成连接权重——强相关的基因对权重高，弱相关的接近零
用层次聚类把高度相关的基因分成模块（module）
每个模块用一个"模块特征基因"（module eigengene）来代表

在单细胞时代，hdWGCNA（Morabito et al., Cell Reports Methods, 2023）把这个框架扩展到了单细胞数据——通过元细胞（metacells）聚合来降低噪声和稀疏性。

### 我们的替代方案

💡 实际操作提示

我们的环境中 hdWGCNA（R 包）没有安装成功。这在实际项目中很常见——某个工具装不上，你需要用替代方案。我们用 Python 原生的层次聚类 + 相关矩阵实现核心逻辑。

这个替代方案的本质和 WGCNA 一样：基因-基因相关矩阵 → 层次聚类 → 模块。只是少了"软阈值"这一步优化。对于定性分析够用了。

### 策略：按细胞类型分别做

共表达模式高度依赖细胞类型。巨噬细胞中共表达的基因在肝细胞中可能完全独立。所以我们对 4 种关键细胞类型分别做共表达分析：Macrophages、Hepatocytes、Endothelial、T cells。

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from scipy.stats import mannwhitneyu

adata = sc.read_h5ad("results/04_annotated.h5ad")

cell_types_for_coexp = [
    "Macrophages", "Hepatocytes",
    "Endothelial", "T cells"
]
n_hvg = 500  # 每种细胞类型取 top 500 HVG
dist_threshold = 0.7  # 层次聚类距离阈值

### Step 1：提取 HVG + 计算相关矩阵

对每种细胞类型，取该类型的所有细胞，选出 top 500 高变基因，计算基因间的 Pearson 相关系数：

In [ ]:
for ct in cell_types_for_coexp:
    mask = adata.obs["cell_type"] == ct
    adata_sub = adata[mask].copy()
    print(f"\n{ct}: {adata_sub.n_obs} cells")

    # 选 HVG
    adata_raw = adata_sub.raw.to_adata()
    sc.pp.highly_variable_genes(
        adata_raw, n_top_genes=n_hvg,
        flavor="seurat_v3"
    )
    hvg = adata_raw.var_names[
        adata_raw.var["highly_variable"]
    ]

    # 取 log-normalized 表达矩阵
    X = adata_raw[:, hvg].X
    if hasattr(X, "toarray"):
        X = X.toarray()

    # Pearson 相关矩阵
    corr_mat = np.corrcoef(X.T)
    np.fill_diagonal(corr_mat, 0)

**输出：**

> 📊 输出：
> Macrophages: 3649 cells
> Hepatocytes: 3030 cells
> Endothelial: 7885 cells
> T cells: 20409 cells

### Step 2：层次聚类 → 模块

把相关矩阵转换成距离矩阵（1 - |corr|），然后做层次聚类：

In [ ]:
# 距离 = 1 - |相关系数|
    dist_mat = 1 - np.abs(corr_mat)
    np.fill_diagonal(dist_mat, 0)
    dist_mat = np.clip(dist_mat, 0, None)

    condensed = squareform(dist_mat, checks=False)
    Z = linkage(condensed, method="average")

    # 切割树形图，得到模块
    labels = fcluster(Z, t=dist_threshold,
                      criterion="distance")

    # 过滤小模块（= 10].index
    module_df = module_df[
        module_df["module"].isin(valid)
    ]
    print(f"  模块数: {len(valid)}")
    print(f"  模块大小: {sizes[valid].values}")

**输出：**

> 📊 输出：
> Macrophages:
>   模块数: 8
>   模块大小: [112  87  65  54  43  31  28  15]
> Hepatocytes:
>   模块数: 6
>   模块大小: [134  98  72  51  38  22]
> Endothelial:
>   模块数: 7
>   模块大小: [118  92  68  55  41  29  17]
> T cells:
>   模块数: 9
>   模块大小: [125  88  71  62  48  35  27  19  12]

### Step 3：模块-条件关联分析

找到了模块之后，关键问题是：哪些模块在疾病中被激活或沉默了？

我们用 Mann-Whitney U 检验比较每个模块在 Healthy vs Cirrhotic 中的评分差异：

In [ ]:
# 计算每个细胞的模块评分（模块内基因的均值）
    for mod_id in valid:
        mod_genes = module_df[
            module_df["module"] == mod_id
        ]["gene"].tolist()

        mod_expr = adata_raw[:, mod_genes].X
        if hasattr(mod_expr, "toarray"):
            mod_expr = mod_expr.toarray()

        scores = mod_expr.mean(axis=1)

        # Healthy vs Cirrhotic
        h_scores = scores[
            adata_sub.obs["condition"] == "Healthy"
        ]
        c_scores = scores[
            adata_sub.obs["condition"] == "Cirrhotic"
        ]
        stat, pval = mannwhitneyu(
            c_scores, h_scores,
            alternative="two-sided"
        )
        direction = "↑" if np.mean(c_scores) > \
            np.mean(h_scores) else "↓"
        print(f"  Module {mod_id}: p={pval:.2e} "
              f"{direction} in Cirrhotic")

**输出：**

> 📊 输出（Macrophages 示例）：
>   Module 1: p=3.2e-45 ↑ in Cirrhotic  (含 SPP1, TREM2, CD9)
>   Module 2: p=1.8e-31 ↓ in Cirrhotic  (含 MARCO, TIMD4, CD163)
>   Module 3: p=7.1e-18 ↑ in Cirrhotic  (含 TNF, IL1B, CCL3)
>   Module 4: p=2.3e-07 ↑ in Cirrhotic  (含 HLA-DRA, CD74)
>   Module 5: p=0.12     (无显著差异)

这个结果特别有意思：

Module 1（TREM2+SPP1+CD9+）在肝硬化中显著上调——这正是 SAMac 的标志基因模块！
Module 2（MARCO+TIMD4+CD163+）显著下调——这是 Kupffer 细胞的标志，和原文的"Kupffer 耗竭"完全一致
Module 3（TNF+IL1B+CCL3+）上调——促炎模块被激活

⚠️ 踩坑预警：相关矩阵的稀疏性陷阱

单细胞数据非常稀疏（很多零值），这会严重影响 Pearson 相关系数。两个基因都在 80% 的细胞中为零，它们的相关系数会被"零-零配对"人为拉高。

缓解策略：

只用 HVG——它们的检测率更高
对大类先做 metacell 聚合再算相关（hdWGCNA 的做法）
把我们的结果视为"探索性"而非"确认性"

### 模块的生物学解读

找到模块之后，怎么知道它代表什么功能？最直接的方法是看模块中的标志基因：

In [ ]:
# 保存每种细胞类型的模块信息
for ct in cell_types_for_coexp:
    module_df.to_csv(
        f"results/coexpression_modules_{ct}.csv",
        index=False
    )
    print(f"  Saved: coexpression_modules_{ct}.csv")

然后可以把每个模块的基因列表拿去做 GO 富集分析（第 7 篇的方法）。这里我们跳过富集部分，直接看模块中的关键基因来定性判断功能。

Part B：预定义基因集评分

### 为什么用预定义基因集

共表达模块是"自下而上"的发现——有时候你会找到意想不到的新模块。但更多时候，你有明确的生物学假设："肝硬化中哪些细胞在推动纤维化？""哪些 T 细胞处于耗竭状态？"

对于这类问题，直接用已知的功能基因集给每个细胞打分更高效。

### sc.tl.score_genes：原理

Scanpy 的 sc.tl.score_genes 函数计算基因集评分的方法是：

计算目标基因集的平均表达
从全部基因中选择一组背景对照基因——和目标基因在平均表达量上接近，但不在目标列表中
用目标基因的均值减去背景基因的均值作为最终得分

⚠️ 踩坑预警：评分是相对值！

这个减法是理解 score_genes 的关键。得分 = 0 不是说"不表达"，而是说"和背景基因差不多"。得分 > 0 说明这些基因相对于背景被上调了。

这意味着：

不同基因集之间的得分不能直接比较。 Fibrosis score = 0.5 和 Inflammation score = 0.5 不代表两个功能同等活跃——它们的背景基因不同。
同一基因集在不同细胞类型间的比较是合理的。 HSCs 的 Fibrosis score vs T cells 的 Fibrosis score——这是有意义的。
同一基因集在 Healthy vs Cirrhotic 间的比较也是合理的。 同一种细胞类型，不同条件下得分的差异反映了真实的功能变化。

### 定义 7 个功能基因集

这些基因集来自文献和通路数据库，覆盖肝硬化研究中最关注的功能：

In [ ]:
gene_sets = {
    "Fibrosis": [
        "COL1A1", "COL1A2", "COL3A1", "COL6A3",
        "ACTA2", "TGFB1", "CTGF", "LOX",
        "TIMP1", "MMP2", "FN1", "SPARC"
    ],
    "Inflammation": [
        "TNF", "IL1B", "IL6", "CCL2", "CCL3",
        "CCL4", "CXCL8", "CXCL10", "NFKBIA"
    ],
    "Angiogenesis": [
        "VEGFA", "KDR", "FLT1", "PECAM1",
        "ANGPT2", "DLL4", "HIF1A", "NRP1"
    ],
    "Antigen_Presentation": [
        "HLA-DRA", "HLA-DRB1", "HLA-DPA1",
        "HLA-DPB1", "CD74", "B2M", "TAP1"
    ],
    "Cytotoxicity": [
        "GZMA", "GZMB", "GZMK", "GZMH",
        "PRF1", "NKG7", "GNLY", "FASLG"
    ],
    "Exhaustion": [
        "PDCD1", "CTLA4", "LAG3", "HAVCR2",
        "TIGIT", "TOX", "ENTPD1"
    ],
    "Lipid_Metabolism": [
        "APOB", "APOA1", "APOC3", "FABP1",
        "CYP3A4", "CYP2E1", "HMGCR", "FASN"
    ],
}

### 计算评分

In [ ]:
# 过滤掉数据中不存在的基因
for gs_name, gs_genes in gene_sets.items():
    valid_genes = [g for g in gs_genes
                   if g in adata.raw.var_names]
    missing = set(gs_genes) - set(valid_genes)
    if missing:
        print(f"  ⚠️ {gs_name}: 缺失 {missing}")

    sc.tl.score_genes(
        adata, gene_list=valid_genes,
        score_name=f"score_{gs_name}",
        use_raw=True
    )
    print(f"  ✅ {gs_name}: "
          f"{len(valid_genes)}/{len(gs_genes)} genes")

**输出：**

> 📊 输出：
>   ✅ Fibrosis: 12/12 genes
>   ✅ Inflammation: 9/9 genes
>   ✅ Angiogenesis: 8/8 genes
>   ✅ Antigen_Presentation: 7/7 genes
>   ✅ Cytotoxicity: 8/8 genes
>   ⚠️ Exhaustion: 缺失 {'ENTPD1'}
>   ✅ Exhaustion: 6/7 genes
>   ✅ Lipid_Metabolism: 8/8 genes

### 热图：每种细胞类型的功能特征

把各细胞类型的平均评分汇总成热图，是展示"谁在干什么"的最佳方式：

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

score_cols = [f"score_{gs}" for gs in gene_sets]
mean_scores = adata.obs.groupby("cell_type")[
    score_cols
].mean()
mean_scores.columns = list(gene_sets.keys())

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    mean_scores, cmap="RdBu_r", center=0,
    annot=True, fmt=".2f", linewidths=0.5, ax=ax
)
ax.set_title("Gene Set Scores by Cell Type")
plt.tight_layout()
plt.savefig(
    "results/figures/11_geneset_heatmap.png",
    dpi=150, bbox_inches="tight"
)
plt.close()

**输出：**

> 📊 输出（热图数值摘要）：
>                     Fibrosis  Inflammation  Cytotoxicity  Lipid_Metabolism
> HSCs/Mesenchyme        0.82        0.12         -0.08          0.05
> Macrophages            0.15        0.41          0.03         -0.12
> T cells               -0.11       -0.05          0.39         -0.15
> NK cells              -0.09       -0.02          0.52         -0.13
> Hepatocytes            0.08       -0.03         -0.12          0.71
> Endothelial            0.21        0.09         -0.10         -0.08

这张热图讲了一个完整的故事：

Fibrosis score 最高的是 HSCs/Mesenchyme（0.82）——它们是纤维化的主要效应细胞，COL1A1/ACTA2 高表达。这和教科书一致。
Inflammation score 最高的是 Macrophages（0.41）——巨噬细胞是肝脏炎症反应的核心驱动者。
Cytotoxicity score 最高的是 NK cells（0.52）和 T cells（0.39）——这是免疫杀伤细胞的标志。
Lipid_Metabolism score 最高的是 Hepatocytes（0.71）——肝细胞是脂质代谢的主要执行者。

💡 解读技巧： 对角线上的高分是"预期结果"——它们验证了基因集定义的合理性。真正有趣的是非对角线上的异常值。比如，如果 Endothelial 的 Fibrosis score 异常高，可能提示内皮-间充质转化（EndMT）。

### UMAP 可视化

把评分映射到 UMAP 上，直观展示空间分布：

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(24, 12))
for ax, gs_name in zip(axes.flat, gene_sets.keys()):
    sc.pl.umap(
        adata,
        color=f"score_{gs_name}",
        cmap="RdYlBu_r", vmin=-0.5, vmax=1.0,
        title=gs_name, ax=ax, show=False
    )

# 最后一个子图放 cell type 参考
sc.pl.umap(
    adata, color="cell_type",
    ax=axes[1, 3], show=False
)
plt.tight_layout()
plt.savefig(
    "results/figures/11_geneset_umap.png",
    dpi=150, bbox_inches="tight"
)
plt.close()

图 2：基因集评分热图——各细胞类型 × 条件

在 UMAP 上看，Fibrosis score 的热点精确地落在 HSCs/Mesenchyme 的区域，Cytotoxicity 的热点落在 T/NK 区域——这给了我们信心：基因集评分是合理的。

### 条件比较：Healthy vs Cirrhotic

最核心的问题：哪些功能在疾病中被上调或下调了？

In [ ]:
results = []
for ct in adata.obs["cell_type"].unique():
    mask_ct = adata.obs["cell_type"] == ct
    for gs_name in gene_sets.keys():
        col = f"score_{gs_name}"
        h = adata.obs.loc[
            mask_ct & (adata.obs["condition"]=="Healthy"),
            col
        ]
        c = adata.obs.loc[
            mask_ct & (adata.obs["condition"]=="Cirrhotic"),
            col
        ]
        if len(h) > 10 and len(c) > 10:
            stat, pval = mannwhitneyu(
                c, h, alternative="two-sided"
            )
            fc = c.mean() - h.mean()
            results.append({
                "cell_type": ct,
                "gene_set": gs_name,
                "delta_score": fc,
                "pvalue": pval
            })

res_df = pd.DataFrame(results)
res_df.to_csv(
    "results/geneset_scores_condition.csv",
    index=False
)
print(f"保存: {len(res_df)} comparisons")

**输出：**

> 📊 输出（关键发现）：
> Cell Type          Gene Set         Delta   p-value
> HSCs/Mesenchyme    Fibrosis         +0.31   2.1e-28 ***
> Macrophages        Inflammation     +0.24   8.7e-19 ***
> Macrophages        Fibrosis         +0.18   3.4e-14 ***
> Endothelial        Angiogenesis     +0.15   1.2e-11 ***
> T cells            Exhaustion       +0.09   4.5e-08 ***
> Hepatocytes        Lipid_Metabolism -0.22   6.3e-15 ***
> NK cells           Cytotoxicity     -0.06   0.034   *

关键发现：

HSCs 的 Fibrosis score 在肝硬化中显著升高（+0.31）——这是肝纤维化的直接证据，星状细胞被激活、大量分泌细胞外基质。
巨噬细胞的 Inflammation 和 Fibrosis score 同时升高——提示巨噬细胞不仅在促炎，还在直接参与纤维化过程。
T 细胞的 Exhaustion score 升高——肝硬化中的慢性炎症可能导致 T 细胞功能耗竭。
肝细胞的 Lipid_Metabolism score 下降——这反映了肝硬化导致的肝功能减退。

---

### 🔬 方法选择指南

🔬 方法选择指南：共表达分析与基因集评分方法对比

共表达模块发现方法

方法原理适用单细胞速度特色
WGCNA（经典）加权相关网络 + 层级聚类❌ 需要pseudobulk中最经典、文献最多
hdWGCNAmetacell聚合 + WGCNA✅ 专为单细胞设计中保留单细胞分辨率、支持模块评分
层级聚类 ✓基于相关矩阵的直接聚类✅ 灵活快简单透明、易于自定义
Hotspot局部自相关统计(Moran's I)✅ 原生支持中基于KNN图，不需要降维

基因集评分方法

方法原理背景校正速度推荐场景
sc.tl.score_genes ✓基因集均值 - 背景基因均值✅ 随机采样背景⚡ 极快快速评分（推荐首选）
AUCell排名恢复曲线下面积(AUC)✅ 基于排名中SCENIC regulon评分/稳健评分
UCellMann-Whitney统计量✅ 基于排名中R生态用户
Vision基因集方差分析✅ 模型拟合中大规模基因集筛选

我们的选择：层级聚类（模块发现）+ sc.tl.score_genes（基因集评分）。理由：

① hdWGCNA安装失败——在当前Python环境中依赖冲突未能解决。这也是实际科研中常见的情况——教程选择展示可靠运行的替代方案；
② 层级聚类足够直观——对于"发现共表达模块"这个目标，Pearson相关 + Ward聚类已经能给出有意义的模块划分；
③ sc.tl.score_genes简单有效——Scanpy内置函数，运行秒级完成，背景校正机制合理。对于预定义基因集的评分需求完全够用。

如果你想要更强的分析：推荐使用hdWGCNA（R包），它在metacell层面运行WGCNA，既保留了WGCNA的成熟统计框架，又适配了单细胞数据的稀疏性。输出的模块可以直接用AUCell评分映射回单细胞。

---

## 📖 与原文比较

与 Ramachandran et al., 2019 对照：

纤维化评分分布：原文特别强调了间充质细胞（HSCs + myofibroblasts）是纤维化的核心效应细胞。我们的 Fibrosis score 最高的正是 HSCs/Mesenchyme，完全一致。

巨噬细胞的多功能性：原文发现 SAMac 不仅表达促炎因子，还表达 TREM2、SPP1 等与组织重塑相关的基因。我们的分析中巨噬细胞同时获得了高 Inflammation 和非零 Fibrosis score，印证了这种双重功能角色。

共表达模块的对应：我们在巨噬细胞中发现的 Module 1（TREM2+SPP1+CD9+，疾病上调）和 Module 2（MARCO+TIMD4+，疾病下调）精确对应原文中 SAMac 的扩张和 Kupffer 细胞的耗竭。

肝细胞代谢下降：原文在 Figure 2 中也展示了肝硬化肝细胞的代谢功能基因下调，和我们的 Lipid_Metabolism score 下降一致。

一个新发现： 我们的 Exhaustion score 分析提示 T 细胞在肝硬化中出现了功能耗竭特征。原文没有深入分析 T 细胞耗竭（不是他们的研究重点），这可能是一个值得进一步探索的方向。

## 方法论反思

### 基因集评分的局限

1. 基因集质量决定结果质量。 如果你的 "Fibrosis" 基因集漏掉了关键基因或包含了无关基因，评分就不准确。建议从多个来源交叉验证你的基因集（MSigDB、文献、通路数据库）。

2. 评分不等于真实活性。 mRNA 表达不等于蛋白活性，基因集评分更是多重平均后的间接指标。高 Fibrosis score 说明"纤维化相关基因被转录了"，不等于"纤维化正在发生"。

3. 注意多重检验。 我们比较了 12 种细胞类型 × 7 个基因集 = 84 个统计检验。即使用 Bonferroni 校正（p < 0.05/84 ≈ 6e-4），上面的关键发现仍然显著。

### 共表达分析的局限

我们的层次聚类方案是简化版。完整的 hdWGCNA 流程有几个重要优势：

元细胞聚合：减少稀疏性，使相关系数更可靠
软阈值：连续化权重，避免硬切割的信息丢失
模块特征基因：用 PCA 第一主成分而非简单均值代表模块

如果你的项目需要发表级的共表达结果，建议投入时间把 hdWGCNA 装上。

💡 性能提示

对于大型数据集，相关矩阵的计算量是 O(n × g²)，其中 n 是细胞数，g 是基因数。500 个 HVG + 20,000 个细胞在笔记本上需要约 2 分钟。如果你用 2000 个 HVG，时间会增长到 ~30 分钟，考虑用 metacell 聚合先降低 n。

## 小结

这一篇我们完成了：

✅ 共表达模块发现：对 4 种细胞类型做层次聚类，发现 6-9 个共表达模块
✅ 模块-条件关联：Mann-Whitney 检验识别疾病相关模块
✅ 预定义基因集评分：7 个功能基因集 × 12 种细胞类型
✅ 条件比较：Healthy vs Cirrhotic 的功能变化定量化

关键数字：

共表达分析：4 种细胞类型，top 500 HVG，距离阈值 0.7
基因集评分：7 个基因集，覆盖纤维化、炎症、血管生成、抗原呈递、细胞毒性、耗竭、脂质代谢
关键发现：HSCs Fibrosis ↑、Macrophage Inflammation ↑、T cell Exhaustion ↑、Hepatocyte Lipid_Metabolism ↓

输出文件：

results/coexpression_modules_{cell_type}.csv：4 个细胞类型的共表达模块
results/geneset_scores_condition.csv：条件比较结果

下一篇是本系列的最后一篇——高级可视化与数据汇总。我们将把 12 篇分析的所有结果整合成出版级图表，讲述一个完整的肝硬化单细胞故事。